# 01 - Sequence Data and Embeddings

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Identify and describe the major types of **sequence data**: text, time series, and audio
- Explain the limitations of **one-hot encoding** for representing discrete tokens
- Understand **word embeddings** as dense, learned representations that capture semantic meaning
- Use `nn.Embedding` in PyTorch as a lookup table mapping indices to vectors
- Visualize embeddings in 2D and interpret what embedding dimensions capture

## Prerequisites

- Python fundamentals (NumPy, basic PyTorch tensors)
- Familiarity with neural network basics (DL100)
- Understanding of matrix operations and dot products

## Table of Contents

1. [What is Sequence Data?](#1-what-is-sequence-data)
2. [One-Hot Encoding and Its Limitations](#2-one-hot-encoding-and-its-limitations)
3. [Word Embeddings: Dense Learned Representations](#3-word-embeddings-dense-learned-representations)
4. [nn.Embedding: The Lookup Table](#4-nnembedding-the-lookup-table)
5. [Embedding Dimensions and What They Capture](#5-embedding-dimensions-and-what-they-capture)
6. [Code: Create Embeddings for a Small Vocabulary](#6-code-create-embeddings-for-a-small-vocabulary)
7. [Code: Visualize Embeddings in 2D](#7-code-visualize-embeddings-in-2d)
8. [Code: Similar Words Get Similar Embeddings](#8-code-similar-words-get-similar-embeddings)
9. [Exercise: Build an Embedding Layer and Inspect Weights](#9-exercise-build-an-embedding-layer-and-inspect-weights)
10. [Common Mistakes & Debugging Tips](#10-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. What is Sequence Data?

**Sequence data** is data where the **order matters**. Unlike tabular data where rows are independent, each element in a sequence depends on what came before (and sometimes after) it.

### Major types of sequence data

| Type | Example | Element | Typical Length |
|------|---------|---------|----------------|
| **Text** | "The cat sat on the mat" | Words or characters | 10--1000 tokens |
| **Time series** | Stock prices, sensor readings | Numerical measurements | 100--10,000 steps |
| **Audio** | Speech waveform, music | Amplitude samples | 16,000+ samples/sec |
| **DNA** | ATCGGATC... | Nucleotide bases | 100--millions |
| **Video** | Sequence of frames | Image frames | 30 frames/sec |

### Key properties of sequence data

- **Temporal/positional ordering**: shuffling elements changes or destroys meaning
- **Variable length**: sequences can have different numbers of elements
- **Long-range dependencies**: meaning can depend on elements far apart (e.g., subject-verb agreement across a long sentence)

In [ ]:
# Example: text as a sequence of tokens
sentence = "The cat sat on the mat"
tokens = sentence.lower().split()
print(f"Sentence: '{sentence}'")
print(f"Tokens:   {tokens}")
print(f"Length:   {len(tokens)} tokens")
print()

# Example: time series as a sequence of numbers
np.random.seed(42)
time_steps = np.arange(50)
stock_price = 100 + np.cumsum(np.random.randn(50) * 0.5)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Text visualization
axes[0].set_xlim(-0.5, len(tokens) - 0.5)
axes[0].set_ylim(-0.5, 0.5)
for i, tok in enumerate(tokens):
    axes[0].annotate(tok, (i, 0), ha='center', va='center', fontsize=14,
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.8))
    if i < len(tokens) - 1:
        axes[0].annotate('', xy=(i + 0.6, 0), xytext=(i + 0.4, 0),
                         arrowprops=dict(arrowstyle='->', color='gray'))
axes[0].set_title('Text: Sequence of Tokens', fontweight='bold')
axes[0].axis('off')

# Time series visualization
axes[1].plot(time_steps, stock_price, 'b-', linewidth=1.5, marker='o', markersize=2)
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Value')
axes[1].set_title('Time Series: Sequence of Numbers', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. One-Hot Encoding and Its Limitations

Before we can feed text into a neural network, we need to convert words into numbers. The simplest approach is **one-hot encoding**:

- Build a vocabulary of $V$ unique words
- Represent each word as a vector of length $V$ with a single 1 and all other entries 0

$$\text{"cat"} \rightarrow [0, 1, 0, 0, 0, 0]$$
$$\text{"dog"} \rightarrow [0, 0, 1, 0, 0, 0]$$

In [ ]:
# Build a small vocabulary and one-hot encode
vocab_words = ["the", "cat", "dog", "sat", "on", "mat"]
word_to_idx = {w: i for i, w in enumerate(vocab_words)}
V = len(vocab_words)

print(f"Vocabulary size: {V}")
print(f"Word-to-index: {word_to_idx}")
print()

# One-hot encode each word
for word in vocab_words:
    one_hot = np.zeros(V)
    one_hot[word_to_idx[word]] = 1.0
    print(f"  '{word:>3s}' -> {one_hot}")

In [ ]:
# Limitations of one-hot encoding

# 1. Sparse and high-dimensional
real_vocab_size = 50000  # typical vocabulary size
one_hot_memory = real_vocab_size * real_vocab_size * 4  # float32, full matrix
print(f"One-hot for V={real_vocab_size}: each vector has {real_vocab_size} dimensions")
print(f"Embedding matrix memory: {one_hot_memory / 1e9:.1f} GB (identity matrix!)")
print()

# 2. No semantic similarity -- all words are equally distant
cat_vec = np.zeros(V); cat_vec[word_to_idx["cat"]] = 1.0
dog_vec = np.zeros(V); dog_vec[word_to_idx["dog"]] = 1.0
mat_vec = np.zeros(V); mat_vec[word_to_idx["mat"]] = 1.0

# Cosine similarity
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

print(f"Cosine similarity (cat, dog): {cosine_sim(cat_vec, dog_vec):.4f}")
print(f"Cosine similarity (cat, mat): {cosine_sim(cat_vec, mat_vec):.4f}")
print()
print("Problem: 'cat' and 'dog' are semantically similar (both animals),")
print("but one-hot treats them as equally distant as 'cat' and 'mat'.")

### Summary of one-hot limitations

- **Sparse**: most entries are zero -- wasteful in memory and computation
- **High-dimensional**: vector length = vocabulary size (can be 50K--100K+)
- **No semantic meaning**: all pairs of words are orthogonal and equidistant
- **No generalization**: the network cannot leverage word similarity

---
## 3. Word Embeddings: Dense Learned Representations

**Word embeddings** solve all three problems of one-hot encoding:

| Property | One-Hot | Embedding |
|----------|---------|----------|
| Dimensionality | $V$ (e.g., 50,000) | $d$ (e.g., 64--300) |
| Sparsity | Extremely sparse | Dense |
| Semantic meaning | None | Similar words $\rightarrow$ similar vectors |
| Learned | No (fixed) | Yes (trained with the model) |

### How embeddings work

An embedding maps each word index to a **dense vector** of dimension $d$:

$$\text{embed}: \{0, 1, \dots, V-1\} \rightarrow \mathbb{R}^d$$

This is implemented as a **lookup table** -- a matrix $\mathbf{E} \in \mathbb{R}^{V \times d}$ where row $i$ is the embedding for word $i$:

$$\text{embed}(i) = \mathbf{E}[i, :]$$

### Key insight

The embedding matrix $\mathbf{E}$ is **learnable**. During training, words that appear in similar contexts will develop similar embedding vectors. This is the foundation of the famous "king - man + woman = queen" analogies.

---
## 4. nn.Embedding: The Lookup Table

PyTorch's `nn.Embedding` is the standard way to create an embedding layer:

```python
emb = nn.Embedding(num_embeddings=V, embedding_dim=d)
```

- `num_embeddings` ($V$): vocabulary size (number of unique tokens)
- `embedding_dim` ($d$): dimensionality of each embedding vector
- The layer stores a weight matrix of shape $(V, d)$
- Input: tensor of integer indices, Output: corresponding embedding vectors

**Note:** `nn.Embedding` is mathematically equivalent to a one-hot vector multiplied by the weight matrix, but is implemented as an efficient index lookup -- no actual matrix multiplication occurs.

In [ ]:
# nn.Embedding basics
set_global_seed(42)

V = 6   # vocabulary size
d = 4   # embedding dimension

emb = nn.Embedding(num_embeddings=V, embedding_dim=d)

print(f"Embedding layer: {emb}")
print(f"Weight matrix shape: {emb.weight.shape}  (V x d = {V} x {d})")
print(f"\nFull embedding matrix:\n{emb.weight.data}")

In [ ]:
# Look up embeddings for specific word indices
idx = torch.tensor([1])  # word index 1 ("cat")
vec = emb(idx)
print(f"Index: {idx.item()} -> Embedding: {vec.data}")
print(f"Matches row 1 of weight matrix: {torch.allclose(vec, emb.weight[1].unsqueeze(0))}")
print()

# Look up multiple words at once
indices = torch.tensor([0, 1, 3])  # "the", "cat", "sat"
vecs = emb(indices)
print(f"Batch lookup for indices {indices.tolist()}:")
print(f"Output shape: {vecs.shape}  (num_words x embedding_dim)")
print(f"Vectors:\n{vecs.data}")

In [ ]:
# Embedding is equivalent to one-hot @ weight_matrix, but faster
one_hot = torch.zeros(V)
one_hot[1] = 1.0  # word index 1

via_matmul = one_hot @ emb.weight.data
via_lookup = emb(torch.tensor([1])).squeeze(0)

print(f"Via one-hot @ W:    {via_matmul.data}")
print(f"Via nn.Embedding:   {via_lookup.data}")
print(f"Identical: {torch.allclose(via_matmul, via_lookup)}")

---
## 5. Embedding Dimensions and What They Capture

Each dimension in an embedding vector captures some **latent feature** of the word. While individual dimensions are not easily interpretable, the overall pattern encodes:

- **Semantic similarity**: "cat" and "dog" are close; "cat" and "airplane" are far
- **Syntactic roles**: nouns cluster together, verbs cluster together
- **Analogies**: $\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$

### Choosing embedding dimension $d$

| Vocabulary Size | Typical $d$ | Use Case |
|----------------|-------------|----------|
| < 1,000 | 16--32 | Toy tasks, small datasets |
| 1,000--50,000 | 64--256 | Most NLP tasks |
| 50,000+ | 256--512 | Large-scale models |

**Rule of thumb**: $d \approx V^{1/4}$ is a reasonable starting point, but this is often tuned as a hyperparameter.

---
## 6. Code: Create Embeddings for a Small Vocabulary

Let us build a complete example: define a vocabulary, create an embedding layer, and encode a sentence.

In [ ]:
set_global_seed(42)

# Define vocabulary
vocab = {
    "<PAD>": 0, "<UNK>": 1,
    "the": 2, "cat": 3, "dog": 4, "sat": 5,
    "on": 6, "mat": 7, "ran": 8, "fast": 9,
    "bird": 10, "fish": 11, "big": 12, "small": 13,
}
idx_to_word = {v: k for k, v in vocab.items()}
V = len(vocab)
d = 8  # embedding dimension

# Create embedding layer with padding_idx
embedding = nn.Embedding(num_embeddings=V, embedding_dim=d, padding_idx=0)

print(f"Vocabulary size: {V}")
print(f"Embedding dim:   {d}")
print(f"Weight shape:    {embedding.weight.shape}")
print(f"PAD vector (should be zeros): {embedding.weight[0].data}")
print()

# Encode a sentence
sentence = "the cat sat on the mat"
tokens = sentence.split()
indices = torch.tensor([vocab.get(t, vocab["<UNK>"]) for t in tokens])

print(f"Sentence: '{sentence}'")
print(f"Tokens:   {tokens}")
print(f"Indices:  {indices.tolist()}")

# Get embeddings
embedded = embedding(indices)
print(f"\nEmbedded shape: {embedded.shape}  (seq_len x emb_dim)")
print(f"\nEmbedding for 'cat' (index 3):")
print(f"  {embedded[1].data}")

---
## 7. Code: Visualize Embeddings in 2D

Since embeddings live in $d$-dimensional space, we use **PCA** (Principal Component Analysis) to project them down to 2D for visualization.

In [ ]:
set_global_seed(42)

# For a more meaningful visualization, let's create an embedding layer
# and manually set weights to show semantic groupings
V_demo = 12
d_demo = 16
words = ["cat", "dog", "bird", "fish",       # animals
         "run", "walk", "swim", "fly",         # actions
         "red", "blue", "green", "yellow"]     # colors
categories = ["animal"] * 4 + ["action"] * 4 + ["color"] * 4

emb_demo = nn.Embedding(V_demo, d_demo)

# Manually bias embeddings so similar categories cluster together
# (In practice, training does this automatically)
with torch.no_grad():
    # Animals: strong signal in dims 0-3
    emb_demo.weight[0:4, 0:4] += 2.0
    # Actions: strong signal in dims 4-7
    emb_demo.weight[4:8, 4:8] += 2.0
    # Colors: strong signal in dims 8-11
    emb_demo.weight[8:12, 8:12] += 2.0

# Extract weight matrix and reduce to 2D with PCA
W = emb_demo.weight.detach().numpy()
pca = PCA(n_components=2)
W_2d = pca.fit_transform(W)

# Plot
color_map = {"animal": "#e74c3c", "action": "#3498db", "color": "#2ecc71"}
fig, ax = plt.subplots(figsize=(8, 6))

for i, (word, cat) in enumerate(zip(words, categories)):
    ax.scatter(W_2d[i, 0], W_2d[i, 1], c=color_map[cat], s=100, zorder=5)
    ax.annotate(word, (W_2d[i, 0] + 0.05, W_2d[i, 1] + 0.05),
                fontsize=12, fontweight='bold')

# Legend
for cat, col in color_map.items():
    ax.scatter([], [], c=col, s=100, label=cat)
ax.legend(fontsize=11)

ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
ax.set_title('Word Embeddings Projected to 2D (PCA)', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print("Notice how words from the same category cluster together!")

---
## 8. Code: Similar Words Get Similar Embeddings

Let us train a tiny embedding on a context-prediction task and verify that semantically similar words develop similar vectors.

In [ ]:
set_global_seed(42)

# Toy corpus: animals do animal things, vehicles do vehicle things
corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat chased the bird",
    "the dog chased the cat",
    "the bird flew over the mat",
    "the dog ran across the rug",
    "the cat ran across the mat",
    "the bird sat on the rug",
    "a car drove on the road",
    "a truck drove on the road",
    "the car stopped at the light",
    "the truck stopped at the light",
    "a car raced down the road",
    "a truck raced down the road",
]

# Build vocabulary
all_words = []
for sent in corpus:
    all_words.extend(sent.split())
unique_words = sorted(set(all_words))
w2i = {"<PAD>": 0}
for w in unique_words:
    w2i[w] = len(w2i)
i2w = {v: k for k, v in w2i.items()}
V_toy = len(w2i)

print(f"Vocabulary ({V_toy} words): {list(w2i.keys())}")

In [ ]:
# Create skip-gram style training pairs: (center_word, context_word)
window_size = 2
pairs = []

for sent in corpus:
    tokens = sent.split()
    indices = [w2i[t] for t in tokens]
    for i, center in enumerate(indices):
        for j in range(max(0, i - window_size), min(len(indices), i + window_size + 1)):
            if i != j:
                pairs.append((center, indices[j]))

centers = torch.tensor([p[0] for p in pairs])
contexts = torch.tensor([p[1] for p in pairs])

print(f"Training pairs: {len(pairs)}")
print(f"First 5 pairs (center -> context):")
for c, ctx in pairs[:5]:
    print(f"  '{i2w[c]}' -> '{i2w[ctx]}'")

In [ ]:
set_global_seed(42)

# Simple skip-gram model
class SkipGram(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.center_emb = nn.Embedding(vocab_size, emb_dim)
        self.context_emb = nn.Embedding(vocab_size, emb_dim)
    
    def forward(self, center, context):
        c = self.center_emb(center)    # (batch, d)
        ctx = self.context_emb(context) # (batch, d)
        # Dot product -> score
        scores = (c * ctx).sum(dim=-1)  # (batch,)
        return scores

emb_dim = 16
model = SkipGram(V_toy, emb_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()

# Train with positive pairs (label=1) and negative samples (label=0)
n_epochs = 200
losses = []

for epoch in range(n_epochs):
    model.train()
    
    # Positive pairs
    pos_scores = model(centers, contexts)
    pos_labels = torch.ones_like(pos_scores)
    
    # Negative samples: random context words
    neg_contexts = torch.randint(1, V_toy, (len(pairs),))  # skip PAD (idx 0)
    neg_scores = model(centers, neg_contexts)
    neg_labels = torch.zeros_like(neg_scores)
    
    # Combined loss
    all_scores = torch.cat([pos_scores, neg_scores])
    all_labels = torch.cat([pos_labels, neg_labels])
    
    loss = loss_fn(all_scores, all_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Skip-Gram Training Loss', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()
print(f"Final loss: {losses[-1]:.4f}")

In [ ]:
# Check similarity: do animals cluster together? do vehicles?
emb_weights = model.center_emb.weight.detach().numpy()

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

# Similarity matrix for selected words
test_words = ["cat", "dog", "bird", "car", "truck", "road", "mat"]
test_indices = [w2i[w] for w in test_words]
test_embs = emb_weights[test_indices]

print("Cosine Similarity Matrix:")
print(f"{'':>8s}", end="")
for w in test_words:
    print(f"{w:>8s}", end="")
print()

for i, w1 in enumerate(test_words):
    print(f"{w1:>8s}", end="")
    for j, w2 in enumerate(test_words):
        sim = cosine_similarity(test_embs[i], test_embs[j])
        print(f"{sim:>8.3f}", end="")
    print()

print()
print("Key observations:")
print(f"  cat-dog similarity:  {cosine_similarity(emb_weights[w2i['cat']], emb_weights[w2i['dog']]):.3f}")
print(f"  car-truck similarity: {cosine_similarity(emb_weights[w2i['car']], emb_weights[w2i['truck']]):.3f}")
print(f"  cat-car similarity:  {cosine_similarity(emb_weights[w2i['cat']], emb_weights[w2i['car']]):.3f}")

In [ ]:
# Visualize learned embeddings in 2D
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(emb_weights[1:])  # skip PAD
word_list = [i2w[i] for i in range(1, V_toy)]

# Color by category
animal_words = {"cat", "dog", "bird"}
vehicle_words = {"car", "truck"}
action_words = {"sat", "chased", "flew", "ran", "drove", "stopped", "raced"}

fig, ax = plt.subplots(figsize=(10, 7))
for i, word in enumerate(word_list):
    if word in animal_words:
        color = "#e74c3c"
    elif word in vehicle_words:
        color = "#3498db"
    elif word in action_words:
        color = "#2ecc71"
    else:
        color = "#95a5a6"
    ax.scatter(emb_2d[i, 0], emb_2d[i, 1], c=color, s=80, zorder=5)
    ax.annotate(word, (emb_2d[i, 0] + 0.02, emb_2d[i, 1] + 0.02), fontsize=10)

# Legend
for label, color in [("animal", "#e74c3c"), ("vehicle", "#3498db"),
                      ("action", "#2ecc71"), ("other", "#95a5a6")]:
    ax.scatter([], [], c=color, s=80, label=label)
ax.legend(fontsize=11)
ax.set_title('Learned Embeddings (PCA Projection)', fontweight='bold')
ax.set_xlabel('PC 1')
ax.set_ylabel('PC 2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9. Exercise: Build an Embedding Layer and Inspect Weights

**Task:**

1. Create a vocabulary of 20 words (include `<PAD>` and `<UNK>`)
2. Build an `nn.Embedding` layer with `embedding_dim=32` and `padding_idx=0`
3. Encode the sentence `"the quick brown fox jumps over the lazy dog"` into indices (map unknown words to `<UNK>`)
4. Pass the index tensor through your embedding layer
5. Print the output shape and verify the PAD embedding is all zeros
6. Compute the cosine similarity between two words of your choice

```python
# Your code here
set_global_seed(42)

# Step 1: Define vocabulary (dict: word -> index)
my_vocab = {"<PAD>": 0, "<UNK>": 1, ...}  # TODO: add 18 more words

# Step 2: Create embedding layer
# my_emb = nn.Embedding(...)

# Step 3: Encode sentence
# sentence = "the quick brown fox jumps over the lazy dog"
# indices = ...

# Step 4: Get embeddings
# embedded = my_emb(indices)

# Step 5: Verify shape and PAD

# Step 6: Cosine similarity
```

In [ ]:
# Try the exercise yourself before looking at the solution!







### Solution

In [ ]:
# ----- Solution -----
set_global_seed(42)

# Step 1: Define vocabulary
my_vocab = {
    "<PAD>": 0, "<UNK>": 1, "the": 2, "a": 3, "cat": 4,
    "dog": 5, "bird": 6, "sat": 7, "ran": 8, "on": 9,
    "over": 10, "under": 11, "big": 12, "small": 13, "quick": 14,
    "slow": 15, "brown": 16, "fox": 17, "lazy": 18, "jumps": 19,
}
print(f"Vocabulary size: {len(my_vocab)}")

# Step 2: Create embedding layer
my_emb = nn.Embedding(num_embeddings=len(my_vocab), embedding_dim=32, padding_idx=0)
print(f"Embedding weight shape: {my_emb.weight.shape}")

# Step 3: Encode sentence
sentence = "the quick brown fox jumps over the lazy dog"
tokens = sentence.split()
indices = torch.tensor([my_vocab.get(t, my_vocab["<UNK>"]) for t in tokens])
print(f"\nSentence: '{sentence}'")
print(f"Indices:  {indices.tolist()}")

# Step 4: Get embeddings
embedded = my_emb(indices)

# Step 5: Verify shape and PAD
print(f"\nEmbedded shape: {embedded.shape}  (expected: [{len(tokens)}, 32])")
pad_vec = my_emb(torch.tensor([0]))
print(f"PAD vector all zeros: {torch.all(pad_vec == 0).item()}")

# Step 6: Cosine similarity
def cos_sim_torch(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b) + 1e-8)

fox_emb = my_emb(torch.tensor([my_vocab["fox"]])).squeeze()
dog_emb = my_emb(torch.tensor([my_vocab["dog"]])).squeeze()
cat_emb = my_emb(torch.tensor([my_vocab["cat"]])).squeeze()

print(f"\nCosine similarity (fox, dog): {cos_sim_torch(fox_emb, dog_emb).item():.4f}")
print(f"Cosine similarity (fox, cat): {cos_sim_torch(fox_emb, cat_emb).item():.4f}")
print("\nNote: With random initialization, similarities are near zero.")
print("After training, semantically similar words would have higher similarity.")

---
## 10. Common Mistakes & Debugging Tips

**1. Forgetting `padding_idx` in `nn.Embedding`**
- If you use padding (index 0 for `<PAD>`), set `padding_idx=0` so the padding embedding stays at zero and does not receive gradient updates.

**2. Index out of range errors**
- `nn.Embedding(V, d)` accepts indices in $[0, V-1]$. Passing index $\geq V$ causes a runtime error.
- Always map unknown words to `<UNK>` index rather than ignoring them.

**3. Confusing `num_embeddings` and `embedding_dim`**
- `num_embeddings` = vocabulary size (rows)
- `embedding_dim` = vector dimensionality (columns)
- Shape of weight matrix: `(num_embeddings, embedding_dim)`

**4. Not detaching before visualization**
- When extracting embedding weights for PCA/plotting, use `.detach().numpy()` to avoid autograd errors.

**5. Using one-hot when embeddings are available**
- One-hot + linear layer is mathematically identical to `nn.Embedding`, but far less memory-efficient for large vocabularies.

**6. Embedding dimension too large or too small**
- Too small: cannot capture enough semantic information
- Too large: overfitting on small datasets, slower training
- Start with $d \in [32, 128]$ for most tasks and tune from there.

---

**Next notebook:** [02 - RNN From Scratch: Intuition](02_RNN_From_Scratch_Intuition.ipynb) -- we build on embeddings and learn how Recurrent Neural Networks process sequences step by step.